# Annotation Results Analysis

This notebook analyzes the five annotations and computes the evaluation and agreement statistics.


In [2]:
from pathlib import Path
from collections import Counter
from itertools import combinations
import json

import pandas as pd
from openpyxl import load_workbook


base_dir = Path("/content")
answers_dir = base_dir
reference_path = base_dir / "narratives_clean.xlsx"
template_path = base_dir / "annotation_sheet.xlsx"

annotation_paths = [answers_dir / f"annotation_{i}.xlsx" for i in range(1, 6)]
categories = ["extracted", "random", "false", "none", "other"]


def normalize(value):
    if value is None:
        return ""
    text = str(value)
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return " ".join(text.split()).strip().lower()


def classify(choice, ref):
    c = normalize(choice)
    if c == normalize(ref["narrative"]):
        return "extracted"
    if c == normalize(ref["random_narrative"]):
        return "random"
    if c == normalize(ref["false_narrative"]):
        return "false"
    if c == normalize("None of the above"):
        return "none"
    return "other"


def cohen_kappa(a, b, labels):
    n = len(a)
    observed = sum(x == y for x, y in zip(a, b)) / n
    counts_a = Counter(a)
    counts_b = Counter(b)
    expected = sum((counts_a[label] / n) * (counts_b[label] / n) for label in labels)
    kappa = 1.0 if expected == 1 else (observed - expected) / (1 - expected)
    return observed, expected, kappa


def fleiss_kappa(matrix, labels):
    n_items = len(matrix)
    n_raters = len(matrix[0])
    category_counts = Counter()
    observed = 0.0
    for row in matrix:
        row_counts = Counter(row)
        category_counts.update(row)
        observed += sum(row_counts[label] * (row_counts[label] - 1) for label in labels) / (n_raters * (n_raters - 1))
    observed /= n_items
    expected = sum((category_counts[label] / (n_items * n_raters)) ** 2 for label in labels)
    kappa = 1.0 if expected == 1 else (observed - expected) / (1 - expected)
    prevalence = {label: category_counts[label] / (n_items * n_raters) for label in labels}
    return observed, expected, kappa, prevalence


def krippendorff_alpha_nominal(matrix, labels):
    observed_disagreements = 0
    observed_pairs = 0
    total_counts = Counter()
    total = 0
    for row in matrix:
        row_counts = Counter(row)
        total_counts.update(row)
        total += len(row)
        observed_pairs += len(row) * (len(row) - 1)
        observed_disagreements += len(row) * (len(row) - 1) - sum(row_counts[label] * (row_counts[label] - 1) for label in labels)
    dobs = observed_disagreements / observed_pairs
    expected_pairs = total * (total - 1)
    expected_agreements = sum(total_counts[label] * (total_counts[label] - 1) for label in labels)
    dexp = (expected_pairs - expected_agreements) / expected_pairs
    return 1.0 if dexp == 0 else 1 - dobs / dexp


def agreement_pattern(row):
    counts = sorted(Counter(row).values(), reverse=True)
    if len(counts) == 1:
        return f"{len(row)}_of_{len(row)}"
    return "_".join(str(x) for x in counts) + "_split"


def mean_sd_sem_ci(values):
    n = len(values)
    mean = sum(values) / n
    sd = (sum((x - mean) ** 2 for x in values) / (n - 1)) ** 0.5 if n > 1 else 0.0
    sem = sd / (n ** 0.5)
    ci_half_width = 1.96 * sem
    return {
        "mean": mean,
        "sd": sd,
        "sem": sem,
        "ci95_low": mean - ci_half_width,
        "ci95_high": mean + ci_half_width,
    }


reference_df = pd.read_excel(reference_path)
refs = reference_df.loc[:99, ["narrative", "random_narrative", "false_narrative"]].to_dict("records")

template_wb = load_workbook(template_path, data_only=False)
template_ws = template_wb.active
template_options = {
    row: [template_ws.cell(row=row, column=col).value for col in range(19, 23)]
    for row in range(2, 102)
}

annotators = []
validation = []

for path in annotation_paths:
    wb = load_workbook(path, data_only=False)
    ws = wb.active
    labels = []
    comments = []
    invalid_rows = []
    changed_option_rows = []

    for i, row in enumerate(range(2, 102)):
        choice = ws.cell(row=row, column=4).value
        comment = ws.cell(row=row, column=5).value
        workbook_options = [ws.cell(row=row, column=col).value for col in range(19, 23)]
        label = classify(choice, refs[i])
        labels.append(label)

        if normalize(comment):
            comments.append(row)

        if normalize(choice) not in {normalize(x) for x in workbook_options}:
            invalid_rows.append(row)

        if [normalize(x) for x in workbook_options] != [normalize(x) for x in template_options[row]]:
            changed_option_rows.append(row)

    annotators.append({
        "name": path.stem,
        "labels": labels,
        "counts": Counter(labels),
        "comments": comments,
    })
    validation.append({
        "file": path.name,
        "rows_checked": len(labels),
        "invalid_selections": len(invalid_rows),
        "dropdown_rows_changed_from_template": len(changed_option_rows),
        "comments": len(comments),
    })

per_annotator = []
for annotator in annotators:
    labels = annotator["labels"]
    counts = annotator["counts"]
    per_annotator.append({
        "annotator": annotator["name"],
        "extracted": counts["extracted"],
        "random": counts["random"],
        "false": counts["false"],
        "none": counts["none"],
        "other": counts["other"],
        "comments": len(annotator["comments"]),
        "extracted_selection_rate": counts["extracted"] / len(labels),
    })

pairwise = []
for a, b in combinations(annotators, 2):
    observed, expected, kappa = cohen_kappa(a["labels"], b["labels"], categories)
    pairwise.append({
        "pair": f'{a["name"]} vs {b["name"]}',
        "raw_agreement": observed,
        "expected_agreement": expected,
        "cohen_kappa": kappa,
        "agree_count": sum(x == y for x, y in zip(a["labels"], b["labels"])),
    })

matrix = [[annotator["labels"][i] for annotator in annotators] for i in range(100)]
fleiss_observed, fleiss_expected, fleiss, label_prevalence = fleiss_kappa(matrix, categories)
alpha = krippendorff_alpha_nominal(matrix, categories)
patterns = Counter(agreement_pattern(row) for row in matrix)
rates = [row["extracted_selection_rate"] for row in per_annotator]
pair_kappas = [row["cohen_kappa"] for row in pairwise]
pair_raw = [row["raw_agreement"] for row in pairwise]
category_stats = {
    category: mean_sd_sem_ci([row[category] for row in per_annotator])
    for category in ["extracted", "random", "false", "none"]
}
rate_stats = mean_sd_sem_ci(rates)
item_majority = []
for row in matrix:
    counts = Counter(row)
    top_count = max(counts.values())
    top_labels = sorted([label for label, count in counts.items() if count == top_count])
    item_majority.append({
        "label": top_labels[0] if len(top_labels) == 1 else "tie",
        "count": top_count,
        "tie": len(top_labels) > 1,
    })
majority_counts = Counter(item["label"] for item in item_majority)


In [3]:
validation_df = pd.DataFrame(validation)
validation_df


,file,rows_checked,invalid_selections,dropdown_rows_changed_from_template,comments
0,annotation_1.xlsx,100,0,0,0
1,annotation_2.xlsx,100,0,0,0
2,annotation_3.xlsx,100,0,0,0
3,annotation_4.xlsx,100,0,0,0
4,annotation_5.xlsx,100,0,0,0


In [4]:
per_annotator_df = pd.DataFrame(per_annotator)
per_annotator_df["extracted_selection_rate"] = (per_annotator_df["extracted_selection_rate"] * 100).round(1)
per_annotator_df


,annotator,extracted,random,false,none,other,comments,extracted_selection_rate
0,annotation_1,87,2,5,6,0,0,87.0
1,annotation_2,83,6,3,8,0,0,83.0
2,annotation_3,83,4,6,7,0,0,83.0
3,annotation_4,88,3,4,5,0,0,88.0
4,annotation_5,82,7,6,5,0,0,82.0


In [5]:
category_summary = pd.DataFrame([
    {"category": category, **stats}
    for category, stats in category_stats.items()
])

for column in ["mean", "sd", "sem", "ci95_low", "ci95_high"]:
    category_summary[column] = category_summary[column].round(2)

category_summary


,category,mean,sd,sem,ci95_low,ci95_high
0,extracted,84.6,2.70,1.21,82.23,86.97
1,random,4.4,2.07,0.93,2.58,6.22
2,false,4.8,1.30,0.58,3.66,5.94
3,none,6.2,1.30,0.58,5.06,7.34


In [6]:
majority_vote_df = pd.DataFrame([
    {"majority_label": key, "items": value}
    for key, value in sorted(majority_counts.items())
])

majority_vote_df


,majority_label,items
0,extracted,86
1,false,4
2,none,6
3,random,4


In [7]:
pairwise_df = pd.DataFrame(pairwise)
pairwise_df["raw_agreement"] = (pairwise_df["raw_agreement"] * 100).round(1)
pairwise_df["expected_agreement"] = (pairwise_df["expected_agreement"] * 100).round(1)
pairwise_df["cohen_kappa"] = pairwise_df["cohen_kappa"].round(3)
pairwise_df

,pair,raw_agreement,expected_agreement,cohen_kappa,agree_count
0,annotation_1 vs annotation_2,86.0,73.0,0.482,86
1,annotation_1 vs annotation_3,90.0,73.0,0.629,90
2,annotation_1 vs annotation_4,91.0,77.1,0.607,91
3,annotation_1 vs annotation_5,85.0,72.1,0.463,85
4,annotation_2 vs annotation_3,92.0,69.9,0.734,92
5,annotation_2 vs annotation_4,87.0,73.7,0.505,87
6,annotation_2 vs annotation_5,92.0,69.1,0.741,92
7,annotation_3 vs annotation_4,91.0,73.8,0.657,91
8,annotation_3 vs annotation_5,91.0,69.0,0.709,91
9,annotation_4 vs annotation_5,90.0,72.9,0.632,90


In [8]:
summary_df = pd.DataFrame([{
    "extracted_narrative_selection_rate": sum(rates) / len(rates) * 100,
    "sd_extracted_selection_points": (sum((x - sum(rates) / len(rates)) ** 2 for x in rates) / len(rates)) ** 0.5 * 100,
    "sem_extracted_selection_points": rate_stats["sem"] * 100,
    "ci95_extracted_selection_low": rate_stats["ci95_low"] * 100,
    "ci95_extracted_selection_high": rate_stats["ci95_high"] * 100,
    "mean_pairwise_raw_agreement": sum(pair_raw) / len(pair_raw) * 100,
    "mean_pairwise_cohen_kappa": sum(pair_kappas) / len(pair_kappas),
    "min_pairwise_cohen_kappa": min(pair_kappas),
    "max_pairwise_cohen_kappa": max(pair_kappas),
    "fleiss_kappa": fleiss,
    "fleiss_observed_agreement": fleiss_observed * 100,
    "fleiss_expected_agreement": fleiss_expected * 100,
    "krippendorff_alpha_nominal": alpha,
}])

summary_display = summary_df.copy()
for column in [
    "extracted_narrative_selection_rate",
    "ci95_extracted_selection_low",
    "ci95_extracted_selection_high",
    "mean_pairwise_raw_agreement",
    "fleiss_observed_agreement",
    "fleiss_expected_agreement",
]:
    summary_display[column] = summary_display[column].round(1)
for column in [
    "mean_pairwise_cohen_kappa",
    "min_pairwise_cohen_kappa",
    "max_pairwise_cohen_kappa",
    "fleiss_kappa",
    "krippendorff_alpha_nominal",
]:
    summary_display[column] = summary_display[column].round(3)
summary_display["sd_extracted_selection_points"] = summary_display["sd_extracted_selection_points"].round(2)
summary_display["sem_extracted_selection_points"] = summary_display["sem_extracted_selection_points"].round(2)
summary_display


,extracted_narrative_selection_rate,sd_extracted_selection_points,sem_extracted_selection_points,ci95_extracted_selection_low,ci95_extracted_selection_high,mean_pairwise_raw_agreement,mean_pairwise_cohen_kappa,min_pairwise_cohen_kappa,max_pairwise_cohen_kappa,fleiss_kappa,fleiss_observed_agreement,fleiss_expected_agreement,krippendorff_alpha_nominal
0,84.6,2.42,1.21,82.2,87.0,89.5,0.616,0.463,0.741,0.62,89.5,72.4,0.621


In [9]:
item_agreement_distribution = pd.DataFrame(
    [{"pattern": key, "items": value} for key, value in sorted(patterns.items())]
)

label_prevalence_table = pd.DataFrame(
    [{"label": key, "prevalence": value} for key, value in label_prevalence.items()]
)

item_agreement_distribution, label_prevalence_table


(       pattern  items
 0  3_1_1_split      1
 1    3_2_split      9
 2    4_1_split     11
 3       5_of_5     79,
        label  prevalence
 0  extracted       0.846
 1     random       0.044
 2      false       0.048
 3       none       0.062
 4      other       0.000)

In [10]:
result = {
    "validation": validation,
    "per_annotator": [
        {**row, "extracted_selection_rate": round(row["extracted_selection_rate"] * 100, 1)}
        for row in per_annotator
    ],
    "pairwise": [
        {
            "pair": row["pair"],
            "raw_agreement": round(row["raw_agreement"] * 100, 1),
            "expected_agreement": round(row["expected_agreement"] * 100, 1),
            "cohen_kappa": round(row["cohen_kappa"], 3),
            "agree_count": row["agree_count"],
        }
        for row in pairwise
    ],
    "summary": {
        "extracted_narrative_selection_rate": round(sum(rates) / len(rates) * 100, 1),
        "sd_extracted_selection_points": round((sum((x - sum(rates) / len(rates)) ** 2 for x in rates) / len(rates)) ** 0.5 * 100, 2),
        "sem_extracted_selection_points": round(rate_stats["sem"] * 100, 2),
        "ci95_extracted_selection_low": round(rate_stats["ci95_low"] * 100, 1),
        "ci95_extracted_selection_high": round(rate_stats["ci95_high"] * 100, 1),
        "mean_pairwise_raw_agreement": round(sum(pair_raw) / len(pair_raw) * 100, 1),
        "mean_pairwise_cohen_kappa": round(sum(pair_kappas) / len(pair_kappas), 3),
        "min_pairwise_cohen_kappa": round(min(pair_kappas), 3),
        "max_pairwise_cohen_kappa": round(max(pair_kappas), 3),
        "fleiss_kappa": round(fleiss, 3),
        "fleiss_observed_agreement": round(fleiss_observed * 100, 1),
        "fleiss_expected_agreement": round(fleiss_expected * 100, 1),
        "krippendorff_alpha_nominal": round(alpha, 3),
    },
    "category_summary_per_annotator": {
        category: {
            "mean": round(stats["mean"], 2),
            "sd": round(stats["sd"], 2),
            "sem": round(stats["sem"], 2),
            "ci95_low": round(stats["ci95_low"], 2),
            "ci95_high": round(stats["ci95_high"], 2),
        }
        for category, stats in category_stats.items()
    },
    "majority_vote_items": dict(sorted(majority_counts.items())),
    "item_agreement_distribution": dict(sorted(patterns.items())),
    "label_prevalence_all_ratings": {key: round(value, 4) for key, value in label_prevalence.items()},
}

print(json.dumps(result, indent=2))


{
  "validation": [
    {
      "file": "annotation_1.xlsx",
      "rows_checked": 100,
      "invalid_selections": 0,
      "dropdown_rows_changed_from_template": 0,
      "comments": 0
    },
    {
      "file": "annotation_2.xlsx",
      "rows_checked": 100,
      "invalid_selections": 0,
      "dropdown_rows_changed_from_template": 0,
      "comments": 0
    },
    {
      "file": "annotation_3.xlsx",
      "rows_checked": 100,
      "invalid_selections": 0,
      "dropdown_rows_changed_from_template": 0,
      "comments": 0
    },
    {
      "file": "annotation_4.xlsx",
      "rows_checked": 100,
      "invalid_selections": 0,
      "dropdown_rows_changed_from_template": 0,
      "comments": 0
    },
    {
      "file": "annotation_5.xlsx",
      "rows_checked": 100,
      "invalid_selections": 0,
      "dropdown_rows_changed_from_template": 0,
      "comments": 0
    }
  ],
  "per_annotator": [
    {
      "annotator": "annotation_1",
      "extracted": 87,
      "random": 2,
  